# 01 - Dataset Exploration

First look at the processed ACNE04 dataset: load the annotations, verify the image<->annotation mapping, and analyse image resolution, aspect ratio, format and colour channels.

Run the conversion first if you have not: `python scripts/convert_acne04.py`.

> Read-only: this notebook never modifies the dataset.

In [1]:
# Make the src/ package importable from the notebooks/ folder (CPU-only, no install).
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
pd.set_option('display.max_columns', 30)

from acnevision.data import (
    eda_data, eda_images, eda_severity, eda_lesions, eda_boxes, eda_viz,
)

FIG = Path.cwd().parent / 'outputs' / 'figures'


## Load the summary written by the conversion step

In [2]:
import json
PROCESSED = eda_data.default_processed_dir()
summary = json.loads((PROCESSED / 'dataset_summary.json').read_text())
summary

{'dataset': 'ACNE04',
 'split': 'train',
 'num_shards_processed': 3,
 'shards': ['data-00000-of-00003.arrow',
  'data-00001-of-00003.arrow',
  'data-00002-of-00003.arrow'],
 'total_records': 1457,
 'images_extracted': 1,
 'images_skipped_existing': 1456,
 'images_failed': 0,
 'missing_annotations': 0,
 'available_columns': ['image',
  'file_name',
  'level',
  'severity',
  'lesion_count_label',
  'num_lesions_annotated',
  'lesion_boxes',
  'split',
  'fold_0_split',
  'fold_1_split',
  'fold_2_split',
  'fold_3_split',
  'fold_4_split'],
 'severity_distribution': {'mild': 513,
  'moderate': 633,
  'severe': 182,
  'very_severe': 129},
 'split_distribution': {'test': 292, 'train': 1165},
 'image_width_stats': {'min': 143,
  'max': 5184,
  'mean': 2918.29,
  'count': 1457},
 'image_height_stats': {'min': 155,
  'max': 3456,
  'mean': 3027.75,
  'count': 1457}}

## Load annotations and attach real image sizes

In [3]:
# Load once: per-image table (+ real image sizes) and the per-box table.
df = eda_data.load_dataframe()
df = eda_data.attach_image_sizes(df)
boxes = eda_data.explode_boxes(df)
print('images:', len(df), '| lesion boxes:', len(boxes))
df.head(3)


images: 1457 | lesion boxes: 18983


,image_file,file_name,level,severity,lesion_count_label,num_lesions_annotated,lesion_boxes,split,fold_0_split,fold_1_split,fold_2_split,fold_3_split,fold_4_split,num_boxes,image_path,width,height,mode,img_format,missing,corrupted,aspect_ratio
0,acne_00000.jpg,levle0_0.jpg,0,mild,3,3,"{'xmin': [223, 217, 186], 'ymin': [290, 250, 5...",train,train,train,train,test,train,3,/media/mahabur/af33425f-321f-4960-a4d5-f823d20...,525,700,RGB,JPEG,False,False,0.750000
1,acne_00001.jpg,levle0_1.jpg,0,mild,3,3,"{'xmin': [394, 654, 930], 'ymin': [933, 2160, ...",train,train,train,test,train,train,3,/media/mahabur/af33425f-321f-4960-a4d5-f823d20...,3112,3456,RGB,JPEG,False,False,0.900463
2,acne_00002.jpg,levle0_10.jpg,1,moderate,20,20,"{'xmin': [357, 394, 314, 316, 265, 170, 203, 9...",train,train,train,train,test,train,20,/media/mahabur/af33425f-321f-4960-a4d5-f823d20...,478,460,RGB,JPEG,False,False,1.039130


## 1. Verify image <-> annotation mapping
Every annotation should point at an image that exists on disk, with no duplicates and no orphan images.

In [4]:
from acnevision.data.validate_dataset import validate
report = validate(PROCESSED)
report

{'ok': True,
 'num_annotations': 1457,
 'num_images_on_disk': 1457,
 'unique_filenames': 1457,
 'missing_images': 0,
 'duplicate_filenames': 0,
 'orphan_images': 0,
 'problems': []}

In [5]:
# Quick independent cross-check from the loaded DataFrame
print('missing images:', int(df['missing'].sum()))
print('corrupted images:', int(df['corrupted'].sum()))
print('duplicate filenames:', int(df['image_file'].duplicated().sum()))

missing images: 0
corrupted images: 0
duplicate filenames: 0


## 2. Image analysis
Counts, width/height/aspect-ratio statistics, file format and colour channels.

In [6]:
eda_images.summarize_images(df)

{'total_images': 1457,
 'readable_images': 1457,
 'missing_images': 0,
 'corrupted_images': 0,
 'width': {'min': 143.0, 'max': 5184.0, 'mean': 2918.29, 'median': 3112.0},
 'height': {'min': 155.0, 'max': 3456.0, 'mean': 3027.75, 'median': 3456.0},
 'aspect_ratio': {'min': 0.5623471882640587,
  'max': 1.7783505154639174,
  'mean': 0.97,
  'median': 0.9004629629629629}}

In [7]:
eda_images.format_distribution(df)

,img_format,count
0,JPEG,1457


In [8]:
eda_images.channel_distribution(df)

,mode,count,channels
0,RGB,1457,3


In [9]:
eda_images.resolution_table(df)

,megapixel_range,count
0,"(0.00642, 2.261]",210
1,"(2.261, 4.497]",0
2,"(4.497, 6.734]",2
3,"(6.734, 8.97]",0
4,"(8.97, 11.207]",1107
5,"(11.207, 13.443]",0
6,"(13.443, 15.679]",0
7,"(15.679, 17.916]",138


### Resolution and aspect-ratio charts (saved under outputs/figures/)

In [10]:
eda_viz.plot_resolution_scatter(df, FIG / 'image_resolution_scatter.png')
eda_viz.plot_aspect_ratio_hist(df, FIG / 'image_aspect_ratio_hist.png')
print('saved resolution + aspect-ratio figures')

saved resolution + aspect-ratio figures


**Takeaways:** images are all RGB JPEG, 0 corrupted/missing. Resolutions vary enormously (143-5184 px wide), mostly portrait. This heterogeneity is why the input-size decision in notebook 03 matters.